In [ ]:
import pandas as pd
from lifelines import CoxPHFitter

In [ ]:
factor_data = pd.read_csv("/mofa/mofa_workflow/results_mosaic/03_results/03_Factor_Data_v31_norecurr_bulkhvg_MOFA.csv")

In [ ]:
meta_data = pd.read_csv("/mofa/mofa_workflow/input_data_mosaic/Prepared_Sample_Meta_Data.csv")

In [ ]:
cols_to_merge = ['sample_id', 'os_years', 'os_censor', 'pfs_years', 'pfs_censor','age_at_diagnosis_years', 'mgmt_promoter_methylation','administrative_gender']

# Subset meta_data to only include the selected columns
meta_data_subset = meta_data[cols_to_merge]

# Merge meta_data_subset into factor_data based on 'sample_id'
merged_df = factor_data.merge(meta_data_subset, on='sample_id', how='left')

In [ ]:
merged_df = merged_df.drop(columns=[f'Factor{i}' for i in range(1, 16) if i != 7]) # only include factor 7

In [ ]:
merged_df.set_index('sample_id', inplace=True)

In [ ]:
merged_df

In [ ]:
encoded_df = pd.get_dummies(
    merged_df, 
    columns=["mgmt_promoter_methylation", "administrative_gender"], 
    drop_first=True, 
    dtype=int
)

In [ ]:
os_dataframe = encoded_df.drop(columns=["pfs_years", "pfs_censor"])
pfs_dataframe = encoded_df.drop(columns=["os_years", "os_censor"])

In [ ]:
# Initialize and fit the Cox model
cph = CoxPHFitter()
#cph.fit(pfs_dataframe, duration_col='pfs_years', event_col='pfs_censor')
cph.fit(os_dataframe, duration_col='os_years', event_col='os_censor')

In [ ]:
cph.print_summary()